In [1]:
import cv2
import numpy as np

block1 = cv2.imread("C:\\image1st\\New folder\\block2_image.jpg")
block2 = cv2.imread("C:\\image1st\\New folder\\block1_image.jpg")
gray1 = cv2.cvtColor(block1, cv2.COLOR_BGR2GRAY)
gray2 = cv2.cvtColor(block2, cv2.COLOR_BGR2GRAY)


ret1, thresh1 = cv2.threshold(gray1, 127, 255, cv2.THRESH_BINARY)
ret2, thresh2 = cv2.threshold(gray2, 127, 255, cv2.THRESH_BINARY)
contours1, hierarchy1 = cv2.findContours(thresh1, cv2.RETR_TREE, cv2.CHAIN_APPROX_NONE)
contours2, hierarchy2 = cv2.findContours(thresh2, cv2.RETR_TREE, cv2.CHAIN_APPROX_NONE)

filtered_contours1 = []
filtered_contours2 = []

for c in contours1 and  contours2 :
    area = cv2.contourArea(c)
    if 1000 < area < 80000:  # adjust these values as needed
        filtered_contours1.append(c)
        filtered_contours2.append(c)
        
sorted_contours1 = sorted(filtered_contours1, key=cv2.contourArea, reverse=True)
sorted_contours2 = sorted(filtered_contours2, key=cv2.contourArea, reverse=True)

plants = ['A', 'B', 'C', 'D', 'E', 'F']
defected_plants_block1 = []
defected_plants_block2 = []

for i, cont in enumerate(sorted_contours1[:len(plants)]):
    cv2.drawContours(block1, [cont], -1, (0, 255, 0), 3)
    X, Y = cont[0][0]
    cv2.putText(block1, "p1" + plants[i], (X, Y + 60), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 2)

for i, cont in enumerate(sorted_contours2[:len(plants)]):
    cv2.drawContours(block2, [cont], -1, (0, 255, 0), 3)
    X, Y = cont[0][0]
    cv2.putText(block2, "p1" + plants[i], (X, Y + 60), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 2)



error: OpenCV(4.5.4) ./modules/imgproc/src/color.cpp:182: error: (-215:Assertion failed) !_src.empty() in function 'cvtColor'


In [2]:
import cv2
import numpy as np
def get_plant_contours(image, min_area=1000, max_area=80000):
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    ret, thresh = cv2.threshold(gray, 127, 255, cv2.THRESH_BINARY)
    contours, hierarchy = cv2.findContours(thresh, cv2.RETR_TREE, cv2.CHAIN_APPROX_NONE)
    filtered_contours = [c for c in contours if min_area < cv2.contourArea(c) < max_area]
    sorted_contours = sorted(filtered_contours, key=cv2.contourArea, reverse=True)
    return sorted_contours


def detect_brown_in_contour(image, contour, brown_threshold=5):
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    mask = np.zeros_like(gray)
    cv2.drawContours(mask, [contour], -1, 255, -1)

    plant_region = cv2.bitwise_and(image, image, mask=mask)
    hsv = cv2.cvtColor(plant_region, cv2.COLOR_BGR2HSV)

    lower_brown1 = np.array([10, 100, 20])
    upper_brown1 = np.array([20, 255, 200])
    lower_brown2 = np.array([20, 100, 20])
    upper_brown2 = np.array([30, 255, 200])

    brown_mask = cv2.inRange(hsv, lower_brown1, upper_brown1) + cv2.inRange(hsv, lower_brown2, upper_brown2)
    brown_pixels = cv2.countNonZero(brown_mask)
    total_pixels = cv2.countNonZero(mask)

    brown_ratio = (brown_pixels / total_pixels) * 100 if total_pixels > 0 else 0
    defected = brown_ratio > brown_threshold

    return defected, brown_ratio, mask

def process_block(image_path, prefix, plants):
    image = cv2.imread(image_path)
    sorted_contours = get_plant_contours(image)
    defected_plants = []

    for i, cont in enumerate(sorted_contours[:len(plants)]):
        defected, brown_ratio, mask = detect_brown_in_contour(image, cont)
        print(f"{prefix}{plants[i]} → Brown ratio: {brown_ratio:.2f}%")
        if defected:
            defected_plants.append(plants[i])
            cv2.drawContours(image, [cont], -1, (0, 0, 255), 3)
            X, Y = cont[0][0]
            cv2.putText(image, "DEFECTED", (X, Y + 30), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 165, 255), 2)
        else:
            cv2.drawContours(image, [cont], -1, (0, 255, 0), 2)

    cv2.imshow(f"Block {prefix}", image)
    return defected_plants


plants = ['A', 'B', 'C', 'D', 'E', 'F']

defected_block1 = process_block("C:\\image1st\\New folder\\block1_image.jpg", "P1", plants)
defected_block2 = process_block("C:\\image1st\\New folder\\block2_image.jpg", "P2", plants)

print("Defected Block 1:", defected_block1)
print("Defected Block 2:", defected_block2)

cv2.waitKey(0)
cv2.destroyAllWindows()



error: OpenCV(4.5.4) ./modules/imgproc/src/color.cpp:182: error: (-215:Assertion failed) !_src.empty() in function 'cvtColor'


In [ ]:
import cv2
import numpy as np

def detect_defected_plants(image_path,min_area=1000,max_area=80000):
    image=cv2.imread(image_path)
    gray = cv2.cvtColor(image,cv2.COLOR_BGR2GRAY)

    ret, thresh = cv2.threshold(gray,127,255,cv2.THRESH_BINARY)

    contours, hierarchy = cv2.findContours(thresh, cv2.RETR_TREE,cv2.CHAIN_APPROX_NONE)

    filtered_contours = []
    for c in contours:
        area = cv2.contourArea(c)
        if 1000 < area < 80000:  # adjust these values as needed
            filtered_contours.append(c)

    sorted_contours = sorted(filtered_contours, key=cv2.contourArea, reverse=True)

    plants = ['A', 'B', 'C', 'D', 'E', 'F']
    defected_plants = []

    for i, cont in enumerate(sorted_contours[:len(plants)]):
        cv2.drawContours(image, [cont], -1, (0, 255, 0), 3)
        X, Y = cont[0][0]
        cv2.putText(image, plants[i], (X, Y +5), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 2)
     
    # detect defected plant 
    mask = np.zeros_like(gray)
    cv2.drawContours(mask, [cont],  -1, 255, -1)

    #crop plant area
    plant_region = cv2.bitwise_and(image,image,mask=mask)
    hsv=cv2.cvtColor(plant_region,cv2.COLOR_BGR2HSV)



    lower_brown1 = np.array([10, 100, 20])
    upper_brown1 = np.array([20, 255, 200])
    lower_brown2 = np.array([20, 100, 20])
    upper_brown2 = np.array([30, 255, 200])

    brown_mask1 = cv2.inRange(hsv, lower_brown1, upper_brown1)
    brown_mask2= cv2.inRange(hsv,lower_brown2, upper_brown2)
    brown_mask = brown_mask1 + brown_mask2

    brown_pixels = cv2.countNonZero(brown_mask)
    total_pixels = cv2.countNonZero(mask)

    if total_pixels > 0:
        brown_ratio = (brown_pixels / total_pixels) * 100
        if brown_ratio > 5:  # threshold (tweak this)
            defected_plants.append(plants[i])
    print(defected_plants)



    cv2.imshow("plant Contours",image)
    cv2.waitKey(0)
    cv2.destroyAllWindows()


detect_defected_plants("block1_image.jpg")
detect_defected_plants("block2_image.jpg")



['E']


In [ ]:
import cv2
import cv2.aruco as aruco
import numpy as np

def findAruco(image, marker_size=4, total_markers=250, draw=True):
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    key = getattr(aruco, f'DICT_{marker_size}X{marker_size}_{total_markers}')
    if key is None:
        print("INVALID ARUCO DICTIONAry")
        return None, None

    arucoDict = aruco.getPredefinedDictionary(key)
    aruco_param = aruco.DetectorParameters()
    # DETECT MARKERS
    detector = aruco.ArucoDetector(arucoDict, aruco_param)
    bbox, ids, rejected = detector.detectMarkers(gray)
    if ids is not None:
        ids_new=ids.flatten()
        print("Defected Markers IDs : ", ids_new)
        if draw:
            aruco.drawDetectedMarkers(image, bbox, ids)
        return bbox, ids
    else:
        print("NO ARUCO DICTIONARY")

while True:
    image = cv2.imread("task1a_image[1].jpg")
    if image is None:
        break
    image = cv2.resize(image, (0, 0), fx=0.5, fy=0.5)
    findAruco(image)
    cv2.imshow("aruco image", image)
    cv2.waitKey(0)
    cv2.destroyAllWindows()
    break
cap.release()
cv2.destroyAllWindows()
    

In [ ]:
# all

import cv2
import cv2.aruco as aruco
import numpy as np

def findAruco(image, marker_size=4, total_markers=250, draw=True):
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    key = getattr(aruco, f'DICT_{marker_size}X{marker_size}_{total_markers}')
    if key is None:
        print("INVALID ARUCO DICTIONAry")
        return None, None

    arucoDict = aruco.getPredefinedDictionary(key)
    aruco_param = aruco.DetectorParameters()
    # DETECT MARKERS
    detector = aruco.ArucoDetector(arucoDict, aruco_param)
    bbox, ids, rejected = detector.detectMarkers(gray)
    if ids is not None:
        ids_new=ids.flatten()
        print("Defected Markers IDs : ", ids_new)
        if draw:
            aruco.drawDetectedMarkers(image, bbox, ids)
        return bbox, ids
    else:
        print("NO ARUCO DICTIONARY")


def roi_blocks(image_path):
    block1=image[150:350,450:650]
    block2=image[455:640,462:645]
    cv2.imshow("Block_1 image",block1)
    cv2.imshow("Block_2 image",block2)
    cv2.waitKey(0)
    cv2.destroyAllWindows()

def plant_contours(image_path,min_area=1000,max_area=80000):
    image=cv2.imread(image_path)
    gray = cv2.cvtColor(image,cv2.COLOR_BGR2GRAY)

    ret, thresh = cv2.threshold(gray,127,255,cv2.THRESH_BINARY)

    contours, hierarchy = cv2.findContours(thresh, cv2.RETR_TREE,cv2.CHAIN_APPROX_NONE)

    filtered_contours = []
    for c in contours:
        area = cv2.contourArea(c)
        if 1000 < area < 80000:  # adjust these values as needed
            filtered_contours.append(c)

    sorted_contours = sorted(filtered_contours, key=cv2.contourArea, reverse=True)

    plants = ['A', 'B', 'C', 'D', 'E', 'F']
    defected_plants = []

    for i, cont in enumerate(sorted_contours[:len(plants)]):
        cv2.drawContours(image, [cont], -1, (0, 255, 0), 3)
        X, Y = cont[0][0]
        cv2.putText(image, plants[i], (X, Y +5), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 2)

    cv2.imshow("plant Contours",image)
    cv2.waitKey(0)
    cv2.destroyAllWindows()

plant_contours("block1_image.jpg")
plant_contours("block2_image.jpg")        



while True:
    
    image = cv2.imread("task1a_image.jpg")
    if image is None:
        break
    image = cv2.resize(image, (0, 0), fx=0.5, fy=0.5)
    findAruco(image)
    cv2.imshow("aruco image", image)
    cv2.waitKey(0)
    cv2.destroyAllWindows()
    break

cv2.destroyAllWindows()


    

In [6]:
import cv2
import numpy as np


def detect_defected_plants(image_path, min_area=1000, max_area=80000):
    image = cv2.imread(image_path)
    if image is None:
        print("image not found")
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    ret, thresh = cv2.threshold(gray, 127, 255, cv2.THRESH_BINARY)

    contours, hierarchy = cv2.findContours(thresh, cv2.RETR_TREE, cv2.CHAIN_APPROX_NONE)

    filtered_contours = []
    for c in contours:
        area = cv2.contourArea(c)
        if 1000 < area < 80000:  # adjust these values as needed
            filtered_contours.append(c)

    sorted_contours = sorted(filtered_contours, key=cv2.contourArea, reverse=True)

    plants = ['A', 'B', 'C', 'D', 'E', 'F']
    defected_plants = []

    for i, cont in enumerate(sorted_contours[:len(plants)]):
        cv2.drawContours(image, [cont], -1, (0, 255, 0), 3)
        X, Y = cont[0][0]
        cv2.putText(image, plants[i], (X, Y + 5), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 2)

    # detect defected plant
        mask = np.zeros_like(gray)
        cv2.drawContours(mask, [cont], -1, 255, -1)

    # crop plant area
        plant_region = cv2.bitwise_and(image, image, mask=mask)
        hsv = cv2.cvtColor(plant_region, cv2.COLOR_BGR2HSV)

        lower_brown1 = np.array([10, 100, 20])
        upper_brown1 = np.array([20, 255, 200])
        lower_brown2 = np.array([20, 100, 20])
        upper_brown2 = np.array([30, 255, 200])

        brown_mask1 = cv2.inRange(hsv, lower_brown1, upper_brown1)
        brown_mask2 = cv2.inRange(hsv, lower_brown2, upper_brown2)
        brown_mask = brown_mask1 + brown_mask2

        brown_pixels = cv2.countNonZero(brown_mask)
        total_pixels = cv2.countNonZero(mask)

        if total_pixels > 0:
            brown_ratio = (brown_pixels / total_pixels) * 100
            if brown_ratio > 5:  # threshold (tweak this)
                defected_plants.append(plants[i])
        

        """cv2.imshow("plant Contours", image)
        cv2.waitKey(0)
        cv2.destroyAllWindows()"""
        return defected_plants


block1 = detect_defected_plants("block1_image.jpg")
block2 = detect_defected_plants("block2_image.jpg")
print(f"Defected plant in Block 1 : {block1}")
print(f"Defected plant in Block 2 : {block2}")



Defected plant in Block 1 : ['A']
Defected plant in Block 2 : ['A']
